# Alpamayo batch worker -- CARLA-Pakete aus acarla

Laedt `model_inputs.npz` aus dem angehaengten Dataset (erzeugt von `scripts/build_model_inputs.py` durch die in M2 verifizierte Adapter-Kette) und faehrt jedes Paket mit der in M0 v23 verifizierten Konfiguration (NF4-Backbone, FP16-Expert, KV-lokaler Split). Ausgabe: `m0_results/plans.json` mit einem PlanResult je Paket.

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys

MODEL_ID = 'nvidia/Alpamayo-1.5-10B'
UPSTREAM_URL = 'https://github.com/NVlabs/alpamayo1.5.git'
UPSTREAM_COMMIT = '24179cfa8b2eeaf775e9e21698b23af0f899522d'
WORK = Path('/kaggle/temp/alpamayo_m0')
CACHE = Path('/kaggle/temp/huggingface')
RESULT = Path('/kaggle/working/m0_results/m0_kaggle_attempt.json')
WORK.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
RESULT.parent.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
os.environ['UV_LINK_MODE'] = 'copy'

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
secret_source = 'environment' if hf_token else None
secret_errors = {}
if not hf_token:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for secret_label in (
        'HF_TOKEN',
        'HUGGING_FACE_HUB_TOKEN',
        'HUGGINGFACE_TOKEN',
        'HF_ACCESS_TOKEN',
        'huggingface',
    ):
        try:
            candidate = secret_client.get_secret(secret_label)
        except Exception as exc:
            secret_errors[secret_label] = f'{type(exc).__name__}: {exc}'
            continue
        if candidate:
            hf_token = candidate
            secret_source = f'kaggle:{secret_label}'
            break
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
os.environ['M0_SECRET_SOURCE'] = secret_source or 'none'

print(json.dumps({
    'python_base': platform.python_version(),
    'hf_token_present': bool(hf_token),
    'hf_token_source': secret_source,
    'secret_lookup_errors': secret_errors,
    'working_free_gib': round(shutil.disk_usage('/kaggle/working').free / 1024**3, 1),
    'temp_free_gib': round(shutil.disk_usage('/kaggle/temp').free / 1024**3, 1),
}, indent=2))
if not hf_token:
    print('WARNING: HF_TOKEN is missing. Model files are public, but the PhysicalAI dataset may require accepted access and authentication.')

In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, check=True, env=os.environ.copy())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.7'], check=True)
UV = shutil.which('uv')
assert UV, 'uv executable was not installed'
REPO = WORK / 'alpamayo1.5'
if not (REPO / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', UPSTREAM_URL, str(REPO)])
run(['git', 'fetch', 'origin', UPSTREAM_COMMIT, '--depth', '1'], cwd=REPO)
run(['git', 'checkout', '--detach', UPSTREAM_COMMIT], cwd=REPO)
run([UV, 'sync', '--no-install-package', 'flash-attn'], cwd=REPO)
PYTHON312 = REPO / '.venv/bin/python'
assert PYTHON312.exists(), 'Python 3.12 environment was not created'
# v20b: explizit in das Runner-Env installieren. Ohne --python loeste uv ein anderes Env auf,
# und der Runner fiel still auf FP16 zurueck (bitsandbytes_import_error in Version 22).
run([UV, 'pip', 'install', '--python', str(PYTHON312), 'bitsandbytes>=0.45.0'], cwd=REPO)
run([str(PYTHON312), '-c', "import bitsandbytes, torch; print({'bitsandbytes': bitsandbytes.__version__, 'cuda': torch.version.cuda})"])
run([str(PYTHON312), '-c', "import platform, torch, transformers; print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__})"])

In [ ]:
runner = 'from __future__ import annotations\n\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom importlib import metadata\nfrom pathlib import Path\n\nimport numpy as np\n\nMODEL_ID = os.environ.get(\'M0_MODEL_ID\', \'nvidia/Alpamayo-1.5-10B\')\nUPSTREAM_COMMIT = \'24179cfa8b2eeaf775e9e21698b23af0f899522d\'\nREPO = Path(\'/kaggle/temp/alpamayo_m0/alpamayo1.5\')\nRESULT = Path(\'/kaggle/working/m0_results/m0_kaggle_attempt.json\')\nCACHE = Path(\'/kaggle/temp/huggingface\')\nsys.path.insert(0, str(REPO / \'src\'))\nos.environ[\'HF_HOME\'] = str(CACHE)\nos.environ[\'HUGGINGFACE_HUB_CACHE\'] = str(CACHE / \'hub\')\n# v19: Fragmentierung -- der OOM in v18 meldete 376 MiB reserviert-unbelegt bei 292 MiB Bedarf.\nos.environ.setdefault(\'PYTORCH_CUDA_ALLOC_CONF\', \'expandable_segments:True\')\n\ndef version(name):\n    try:\n        return metadata.version(name)\n    except metadata.PackageNotFoundError:\n        return None\n\ndef safe(value):\n    if isinstance(value, dict):\n        return {str(k): safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [safe(v) for v in value]\n    if isinstance(value, (str, int, float, bool)) or value is None:\n        return value\n    current = value\n    for method in (\'detach\', \'cpu\'):\n        fn = getattr(current, method, None)\n        if fn is not None:\n            current = fn()\n    fn = getattr(current, \'numpy\', None)\n    array = fn() if fn is not None else np.asarray(current)\n    return array.item() if array.ndim == 0 else array.tolist()\n\ndef write_result(result):\n    RESULT.parent.mkdir(parents=True, exist_ok=True)\n    RESULT.write_text(json.dumps(safe(result), indent=2), encoding=\'utf-8\')\n\ndef synthetic_contract_data(torch):\n    """Valid Alpamayo tensor shapes without any gated dataset content."""\n    height, width = 320, 576\n    image_frames = torch.zeros((4, 4, 3, height, width), dtype=torch.uint8)\n    x = torch.linspace(0, 255, width, dtype=torch.uint8).view(1, 1, width)\n    y = torch.linspace(0, 255, height, dtype=torch.uint8).view(1, height, 1)\n    image_frames[:, :, 0] = x\n    image_frames[:, :, 1] = y\n    for camera in range(4):\n        image_frames[camera, :, 2].fill_(camera * 60)\n    identity = torch.eye(3, dtype=torch.float32)\n    return {\n        \'image_frames\': image_frames,\n        \'camera_indices\': torch.tensor([0, 1, 2, 6], dtype=torch.int64),\n        \'ego_history_xyz\': torch.zeros((1, 1, 16, 3), dtype=torch.float32),\n        \'ego_history_rot\': identity.view(1, 1, 1, 3, 3).repeat(1, 1, 16, 1, 1),\n        \'ego_future_xyz\': torch.zeros((1, 1, 64, 3), dtype=torch.float32),\n        \'ego_future_rot\': identity.view(1, 1, 1, 3, 3).repeat(1, 1, 64, 1, 1),\n    }\n\nresult = {\n    \'status\': \'started\',\n    \'model_id\': MODEL_ID,\n    \'upstream_git_revision\': subprocess.check_output(\n        [\'git\', \'-C\', str(REPO), \'rev-parse\', \'HEAD\'], text=True\n    ).strip(),\n    \'expected_upstream_revision\': UPSTREAM_COMMIT,\n    \'attention_backend\': \'sdpa\',\n    \'seed\': 42,\n    \'repeats\': 2,\n    \'num_traj_samples\': 1,\n    \'environment\': {\n        \'python\': platform.python_version(),\n        \'torch\': version(\'torch\'),\n        \'transformers\': version(\'transformers\'),\n        \'accelerate\': version(\'accelerate\'),\n        \'hf_token_present\': bool(os.environ.get(\'HF_TOKEN\')),\n        \'hf_token_source\': os.environ.get(\'M0_SECRET_SOURCE\', \'none\'),\n    },\n}\nwrite_result(result)\n\ntry:\n    import torch\n    from alpamayo1_5 import helper\n    from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset\n    from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is unavailable\')\n    gpus = []\n    for index in range(torch.cuda.device_count()):\n        props = torch.cuda.get_device_properties(index)\n        gpus.append({\n            \'index\': index,\n            \'name\': props.name,\n            \'memory_gib\': props.total_memory / 1024**3,\n            \'compute_capability\': [props.major, props.minor],\n        })\n        torch.cuda.reset_peak_memory_stats(index)\n    result[\'environment\'][\'cuda\'] = torch.version.cuda\n    result[\'environment\'][\'gpus\'] = gpus\n    result[\'environment\'][\'driver\'] = subprocess.run(\n        [\'nvidia-smi\', \'--query-gpu=driver_version\', \'--format=csv,noheader\'],\n        capture_output=True,\n        text=True,\n        check=False,\n    ).stdout.strip().splitlines()\n\n    largest = max(gpu[\'memory_gib\'] for gpu in gpus)\n    aggregate = sum(gpu[\'memory_gib\'] for gpu in gpus)\n    native_bf16 = any(gpu[\'compute_capability\'][0] >= 8 for gpu in gpus)\n    # v20: Speicherstrategie 1 aus dem Projektauftrag. FP16 auf 2x T4 ist widerlegt:\n    # v18 (12/12 GiB) -> OOM auf GPU 1 im SDPA des Reasoning-Rollouts; v19 (12/9 GiB) -> 25 Module\n    # auf CPU ausgelagert und Cross-Device-cat im Upstream-Rollout. FP16-Gewichte ~24 GB auf 29 GB\n    # lassen kein Polster fuer Aktivierungen. NF4 nur auf den LM-Layern des Backbones; Vision-Encoder,\n    # lm_head und der gesamte Diffusions-Expert (der die Zahlen ausgibt) bleiben FP16.\n    requested = os.environ.get(\'M0_STRATEGY\', \'nf4\')\n    try:\n        import bitsandbytes  # noqa: F401\n        bnb_available = True\n    except Exception as exc:\n        bnb_available = False\n        result[\'bitsandbytes_import_error\'] = f\'{type(exc).__name__}: {exc}\'\n    if largest >= 24 and native_bf16:\n        strategy = \'upstream_bf16_single_gpu_sdpa\'\n        dtype = torch.bfloat16\n        load_kwargs = {}\n    elif requested == \'nf4\' and bnb_available and largest >= 14:\n        from transformers import BitsAndBytesConfig\n        # v21: NF4 auf einer GPU (v20) lief in den OOM bei 13.99 GiB -- die FP16-Reste (Embedding,\n        # lm_head, Vision, Expert 4.6 GB) summieren sich auf ~12 GB Gewichte. Deshalb ueber alle GPUs\n        # verteilen, aber OHNE cpu in max_memory: kein stilles Offload (v19 zeigte, dass der\n        # Upstream-Rollout damit nicht laeuft) -- lieber laut scheitern.\n        multi_gpu = len(gpus) >= 2 and os.environ.get(\'M0_NF4_SINGLE_GPU\') != \'1\'\n        # v22: device_map=\'auto\' (v21) schneidet layerweise; der Expert liest den KV-Cache ALLER\n        # VLM-Layer und konkatenierte dann ueber die Geraetegrenze (cache_utils.update, torch.cat).\n        # Deshalb explizit entlang der Datenabhaengigkeit teilen: alle LM-Layer + Expert auf GPU 1\n        # (Cache und Leser auf derselben Karte), Vision + Embedding + lm_head auf GPU 0 (die\n        # 151k-Vokabular-Logits entstehen dort, wo sonst Platz waere).\n        strategy = (\'experimental_nf4_backbone_fp16_expert_kvlocal_split_v23\' if multi_gpu\n                    else \'experimental_nf4_backbone_fp16_expert_single_gpu_v20\')\n        kv_local_map = {\n            \'vlm.model.visual\': 0,\n            \'vlm.model.language_model.embed_tokens\': 0,\n            # v23: lm_head zu den Layern -- der Rollout haelt Logits, Sequenzen und den\n            # Diffusionszustand auf input_ids.device (alpamayo1_5.py:286); alles ab Layer 0\n            # muss deshalb auf EINER Karte liegen. v22 scheiterte in _euler an x(GPU0)+v(GPU1).\n            \'vlm.lm_head\': 1,\n            \'vlm.model.language_model.layers\': 1,\n            \'vlm.model.language_model.norm\': 1,\n            \'vlm.model.language_model.rotary_emb\': 1,\n            \'expert\': 1, \'action_space\': 1, \'diffusion\': 1,\n            \'action_in_proj\': 1, \'action_out_proj\': 1,\n        }\n        dtype = torch.float16\n        skip_modules = [\n            \'visual\',            # vlm.model.visual -- Vision-Encoder\n            \'lm_head\',           # vlm.lm_head -- Reasoning-Token-Logits\n            \'expert\',            # Diffusions-/Flow-Matching-Expert: gibt die Trajektorie aus\n            \'diffusion\', \'action_space\', \'action_in_proj\', \'action_out_proj\',\n        ]\n        load_kwargs = {\n            \'device_map\': kv_local_map if multi_gpu else {\'\': 0},\n            \'quantization_config\': BitsAndBytesConfig(\n                load_in_4bit=True,\n                bnb_4bit_quant_type=\'nf4\',\n                bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n                llm_int8_skip_modules=skip_modules,\n            ),\n        }\n        result[\'quantization\'] = {\n            \'scheme\': \'nf4_double_quant\', \'compute_dtype\': \'float16\',\n            \'skip_modules\': skip_modules,\n        }\n    elif len(gpus) >= 2 and aggregate >= 24:\n        strategy = \'experimental_fp16_device_map_auto_sdpa_asym_v19\'\n        dtype = torch.float16\n        # v19: asymmetrische Kappen. In v18 (12/12 GiB) landeten LM-Layer 20-35, lm_head und der\n        # komplette Diffusions-Expert auf der LETZTEN GPU, die dann bei der Inferenz (Logits ueber das\n        # volle Vokabular, SDPA im Reasoning-Rollout) mit 14.48/14.56 GiB in den OOM lief. Die letzte\n        # GPU bekommt deshalb 3 GiB weniger Gewichte, damit dort Platz fuer Aktivierungen bleibt.\n        last_index = max(gpu[\'index\'] for gpu in gpus)\n        max_memory = {\n            gpu[\'index\']: f"{max(1, int(gpu[\'memory_gib\'] - (5 if gpu[\'index\'] == last_index else 2)))}GiB"\n            for gpu in gpus\n        }\n        max_memory[\'cpu\'] = \'24GiB\'\n        load_kwargs = {\n            \'device_map\': \'auto\',\n            \'max_memory\': max_memory,\n            \'offload_folder\': \'/kaggle/temp/alpamayo_offload\',\n            \'offload_state_dict\': True,\n        }\n    else:\n        result[\'status\'] = \'blocked_by_hardware\'\n        result[\'strategy\'] = \'none\'\n        result[\'block_reason\'] = (\n            \'Need one 24 GiB sm80+ GPU or at least two GPUs with 24 GiB aggregate; \'\n            f"found {len(gpus)} GPU(s), {aggregate:.1f} GiB."\n        )\n        raise SystemExit(2)\n    result[\'strategy\'] = strategy\n    result[\'model_dtype\'] = str(dtype)\n    result[\'strategy_classification\'] = (\n        \'reference\' if strategy.startswith(\'upstream_\') else \'experimental_unverified\'\n    )\n    result[\'max_memory\'] = load_kwargs.get(\'max_memory\')\n    write_result(result)\n\n\n    # ================= BATCH-WORKER: Pakete aus results/<run>/model_inputs.npz ==================\n    import glob as _glob\n    # Diagnose: was ist unter /kaggle/input tatsaechlich gemountet? (Version 1 lief ohne Dataset)\n    mounted = []\n    for root, dirs, files in os.walk(\'/kaggle/input\'):\n        for fn in files[:50]:\n            fp = os.path.join(root, fn); mounted.append({\'path\': fp, \'bytes\': os.path.getsize(fp)})\n        if len(mounted) > 200: break\n    result[\'kaggle_input_listing\'] = mounted\n    write_result(result)\n    npz_candidates = sorted(_glob.glob(\'/kaggle/input/**/model_inputs.npz\', recursive=True))\n    assert npz_candidates, (\'model_inputs.npz in keinem angehaengten Dataset gefunden. Gemountet: \'\n                            + json.dumps([m[\'path\'] for m in mounted][:20]) +\n                            \' -- im Editor unter "Input" das Dataset says43/acarla-model-inputs anhaengen.\')\n    NPZ = npz_candidates[0]\n    meta_path = os.path.join(os.path.dirname(NPZ), \'meta.json\')\n    pk = np.load(NPZ)\n    P = int(pk[\'image_frames\'].shape[0])\n    result[\'input\'] = {\n        \'source\': \'acarla_model_inputs\', \'npz\': NPZ, \'packets\': P,\n        \'image_frames_shape\': list(pk[\'image_frames\'].shape), \'image_frames_dtype\': str(pk[\'image_frames\'].dtype),\n        \'camera_indices\': pk[\'camera_indices\'].tolist(),\n        \'ego_history_xyz_shape\': list(pk[\'ego_history_xyz\'].shape),\n        \'ego_history_rot_shape\': list(pk[\'ego_history_rot\'].shape),\n        \'frame_ids\': pk[\'frame_ids\'].tolist(), \'sim_times\': pk[\'sim_times\'].tolist(),\n        \'run_meta\': json.load(open(meta_path)) if os.path.exists(meta_path) else None,\n    }\n    assert tuple(pk[\'image_frames\'].shape[1:4]) == (4, 4, 3), pk[\'image_frames\'].shape\n    assert tuple(pk[\'camera_indices\'].reshape(-1).tolist()) == (0, 1, 2, 6)\n    assert tuple(pk[\'ego_history_xyz\'].shape[1:]) == (1, 1, 16, 3)\n    write_result(result)\n\n    load_start = time.perf_counter()\n    model = Alpamayo1_5.from_pretrained(\n        MODEL_ID,\n        dtype=dtype,\n        attn_implementation=\'sdpa\',\n        **load_kwargs,\n    )\n    if not load_kwargs:\n        model = model.to(\'cuda\')\n    model.eval()\n    result[\'model_load_seconds\'] = time.perf_counter() - load_start\n    result[\'hf_device_map\'] = safe(getattr(model, \'hf_device_map\', None))\n    result[\'parameter_devices\'] = sorted({str(p.device) for p in model.parameters()})\n    quantized = sorted({n.rsplit(\'.\', 1)[0] for n, m in model.named_modules()\n                        if type(m).__name__ in (\'Linear4bit\', \'Linear8bitLt\')})\n    result[\'quantized_module_count\'] = len(quantized)\n    result[\'quantized_modules_sample\'] = quantized[:6] + ([\'...\'] if len(quantized) > 6 else [])\n    result[\'expert_dtypes\'] = sorted({str(p.dtype) for n, p in model.named_parameters() if n.startswith(\'expert\')})\n    result[\'lm_layer_dtypes\'] = sorted({str(p.dtype) for n, p in model.named_parameters()\n                                        if \'language_model.layers.\' in n})\n\n    processor = helper.get_processor(model.tokenizer)\n    input_device = next(model.expert.parameters()).device\n    result[\'input_device\'] = str(input_device)\n    import hashlib\n    sampling = {\'seed\': 42, \'top_p\': 0.98, \'temperature\': 0.6, \'num_traj_samples\': 1, \'max_generation_length\': 256}\n    cfg_str = json.dumps({\'strategy\': strategy, \'revision\': result[\'upstream_git_revision\'], \'dtype\': str(dtype),\n                          \'sampling\': sampling, \'quantization\': result.get(\'quantization\')}, sort_keys=True)\n    model_config_hash = hashlib.sha256(cfg_str.encode()).hexdigest()[:16]\n    result[\'model_config_hash\'] = model_config_hash\n    result[\'sampling\'] = sampling\n\n    plans = []\n    for p in range(P):\n        frames = torch.from_numpy(pk[\'image_frames\'][p])            # (N_cam, 4, 3, H, W) uint8\n        cam_idx = torch.from_numpy(pk[\'camera_indices\'])\n        messages = helper.create_message(frames=frames.flatten(0, 1), camera_indices=cam_idx)\n        inputs = processor.apply_chat_template(\n            messages, tokenize=True, add_generation_prompt=False,\n            continue_final_message=True, return_dict=True, return_tensors=\'pt\',\n        )\n        model_inputs = helper.to_device({\n            \'tokenized_data\': inputs,\n            \'ego_history_xyz\': torch.from_numpy(pk[\'ego_history_xyz\'][p]),\n            \'ego_history_rot\': torch.from_numpy(pk[\'ego_history_rot\'][p]),\n        }, input_device)\n        torch.manual_seed(sampling[\'seed\']); torch.cuda.manual_seed_all(sampling[\'seed\'])\n        torch.cuda.synchronize(); start = time.perf_counter()\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=dtype):\n            pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(\n                data=model_inputs, top_p=sampling[\'top_p\'], temperature=sampling[\'temperature\'],\n                num_traj_samples=sampling[\'num_traj_samples\'],\n                max_generation_length=sampling[\'max_generation_length\'], return_extra=True,\n            )\n        torch.cuda.synchronize(); dt_s = time.perf_counter() - start\n        xyz = pred_xyz.detach().float().cpu().numpy()[0, 0, 0]     # (64, 3)\n        rot = pred_rot.detach().float().cpu().numpy()[0, 0, 0]     # (64, 3, 3)\n        finite = bool(np.isfinite(xyz).all() and np.isfinite(rot).all())\n        reasoning = safe(extra)\n        cot = None\n        try:\n            cot = reasoning[\'cot\'][0][0][0]\n        except Exception:\n            pass\n        plans.append({\n            \'frame_id\': int(pk[\'frame_ids\'][p]), \'sim_time\': float(pk[\'sim_times\'][p]),\n            \'waypoints_xyz\': xyz.tolist(), \'waypoints_rot\': rot.tolist(),\n            \'reasoning\': cot, \'reasoning_raw\': reasoning, \'inference_ms\': dt_s * 1000.0,\n            \'model_config_hash\': model_config_hash, \'finite\': finite,\n            \'end_xyz\': xyz[-1].tolist(), \'path_length_m\': float(np.linalg.norm(np.diff(xyz[:, :2], axis=0), axis=1).sum()),\n        })\n        plans[-1] = safe(plans[-1])\n        result[\'plans_done\'] = p + 1\n        write_result(result)\n        print(json.dumps({\'packet\': p, \'frame_id\': plans[-1][\'frame_id\'], \'inference_s\': round(dt_s, 1),\n                          \'finite\': finite, \'end_xyz\': [round(float(v), 2) for v in xyz[-1]], \'cot\': cot}))\n        # nach JEDEM Paket persistieren -- Version 3 verlor Paket 1 an einen Fehler in der Ausgabe\n        Path(\'/kaggle/working/m0_results/plans.json\').write_text(json.dumps(plans), encoding=\'utf-8\')\n        del model_inputs, inputs, pred_xyz, pred_rot, extra\n        torch.cuda.empty_cache()\n\n    Path(\'/kaggle/working/m0_results/plans.json\').write_text(json.dumps(plans), encoding=\'utf-8\')\n    result[\'plans_file\'] = \'/kaggle/working/m0_results/plans.json\'\n    result[\'n_plans\'] = len(plans)\n    result[\'all_finite\'] = all(pl[\'finite\'] for pl in plans)\n    result[\'cuda_memory\'] = [{\n        \'index\': index,\n        \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(index),\n        \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(index),\n    } for index in range(torch.cuda.device_count())]\n    result[\'status\'] = \'ok\'\nexcept SystemExit:\n    raise\nexcept Exception as exc:\n    result[\'status\'] = \'failed\'\n    result[\'exception\'] = {\'type\': type(exc).__name__, \'message\': str(exc), \'traceback\': traceback.format_exc()}\nfinally:\n    write_result(result)\n    print(json.dumps({\'status\': result.get(\'status\'), \'strategy\': result.get(\'strategy\'),\n                      \'n_plans\': result.get(\'n_plans\'), \'exception\': (result.get(\'exception\') or {}).get(\'message\')}, indent=2))\nif result.get(\'status\') == \'failed\':\n    raise SystemExit(1)\n'
runner_path = Path('/kaggle/working/alpamayo_m0_runner.py')
runner_path.write_text(runner, encoding='utf-8')
print(f'Wrote {runner_path} ({len(runner.splitlines())} lines)')

In [ ]:
completed = subprocess.run(
    [str(PYTHON312), str(runner_path)], env=os.environ.copy(), check=False
)
print('Runner exit code:', completed.returncode)
if RESULT.exists():
    m0 = json.loads(RESULT.read_text(encoding='utf-8'))
    display({
        'status': m0.get('status'),
        'strategy': m0.get('strategy'),
        'strategy_classification': m0.get('strategy_classification'),
        'input_source': m0.get('input', {}).get('source'),
        'm0_qualified': m0.get('m0_qualified'),
        'm0_passed': m0.get('m0_passed'),
        'gpus': m0.get('environment', {}).get('gpus'),
        'model_load_seconds': m0.get('model_load_seconds'),
        'runs': [
            {k: run.get(k) for k in (
                'repeat', 'inference_seconds', 'min_ade_meters', 'finite'
            )}
            for run in m0.get('runs', [])
        ],
        'max_repeat_delta_meters': m0.get('max_repeat_delta_meters'),
        'dataset_exception': m0.get('dataset_exception'),
        'exception': m0.get('exception'),
    })
else:
    raise FileNotFoundError(RESULT)